# Regular expressions with the `re` module

This notebook practices regular expressions on small texts, then on a corpus of nearly 4,000 fraudulent e-mails:

1. Cleaning up a text copied from a PDF
2. Phone numbers in one format
3. HTML links to Markdown
4. Quoted strings
5. A corpus of fraudulent e-mails

Exercises 1 to 4 are independent; the steps of exercise 5 build on each other. Each exercise has a cell for your code, then additional hints and its solution, collapsed: open them if needed.

To work out a pattern, https://regex101.com/ (with the "Python" flavor) explains each of its parts and shows what it matches. If you need a refresher on the basics, the https://regexone.com/ tutorial goes through them step by step.

## 1. Cleaning up a text copied from a PDF

The text `text`, copied from a PDF, has extra blanks: several spaces or a tab between two words, a space before commas and periods, spaces at the end of lines. With `re.sub`, build `clean`, the text without these blanks, and display it. You should get:

```
The re module searches for patterns.
It replaces, splits and extracts
the pieces of a text.
```

Keep the line breaks: only the spaces and tabs are extra.

In [ ]:
import re

text = (
  "The  re module   searches for patterns .\n"
  "It\treplaces , splits and extracts   \n"
  "the pieces of a text ."
)

# Your code here

### Additional hints

Three replacements, one after the other:

- `[ \t]+` matches a run of spaces and tabs, to replace with a single space. `\s+` would go too far: it also matches the line break `\n`, and would join the lines
- a space followed by a comma or a period: capture the punctuation in a group, and put it back alone with `\1` in the replacement
- the spaces right before the end of a line: `$` matches every end of line only with the `re.MULTILINE` flag, passed to `re.sub` by name, `flags=`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
clean = re.sub(r"[ \t]+", " ", text)
clean = re.sub(r"...", r"\1", clean)
clean = re.sub(r"...", "", clean, flags=...)
print(clean)
```

### Solution

In [ ]:
clean = re.sub(r"[ \t]+", " ", text)
clean = re.sub(r" ([,.])", r"\1", clean)
clean = re.sub(r" +$", "", clean, flags=re.MULTILINE)
print(clean)

## 2. Phone numbers in one format

A form received the French phone numbers of `numbers`, written in six ways. Write a function `normalize(number)` that returns the number in the format `06 39 98 12 34`: ten digits in groups of two, the `+33` country code replaced with a `0`. Then display the normalized number of each item of `numbers`. You should get:

```
06 39 98 12 34
06 39 98 12 34
06 39 98 12 34
06 39 98 12 34
01 99 00 56 78
01 99 00 56 78
```

In [ ]:
numbers = [
  "06 39 98 12 34",
  "06.39.98.12.34",
  "0639981234",
  "+33 6 39 98 12 34",
  "01-99-00-56-78",
  "+33 (0)1 99 00 56 78",
]

# Your code here

### Additional hints

Three replacements in `normalize`:

- `\D` matches what isn't a digit: removing it keeps only the digits, and `"+33 (0)1 99 00 56 78"` becomes `"330199005678"`
- a `33` at the start of the string, followed or not by a `0`, becomes a `0`: `^` anchors the pattern at the start, `?` makes the `0` optional
- five groups of two digits, `(\d{2})` five times, are rewritten `\1 \2 \3 \4 \5`

Example: `normalize("+33 (0)1 99 00 56 78")` returns `"01 99 00 56 78"`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def normalize(number: str) -> str:
  digits = re.sub(r"\D", "", number)
  digits = re.sub(r"...", "0", digits)
  return re.sub(r"...", r"\1 \2 \3 \4 \5", digits)


for number in numbers:
  print(normalize(number))
```

### Solution

In [ ]:
def normalize(number: str) -> str:
  digits = re.sub(r"\D", "", number)
  digits = re.sub(r"^330?", "0", digits)
  return re.sub(r"(\d{2})(\d{2})(\d{2})(\d{2})(\d{2})", r"\1 \2 \3 \4 \5", digits)


for number in numbers:
  print(normalize(number))

## 3. HTML links to Markdown

In HTML, a link is written `<a href="address">text</a>`; in Markdown, `[text](address)`. With `re.sub`, rewrite the two links of `html` in Markdown, and display the result:

```
<p>See [the re docs](https://docs.python.org/3/library/re.html) and [regex101](https://regex101.com/).</p>
```

First try the pattern `<a href="(.*)">(.*)</a>`: why doesn't it give this result? Fix it.

**Useful documentation**

- [Greedy versus Non-Greedy](https://docs.python.org/3/howto/regex.html#greedy-versus-non-greedy)

In [ ]:
html = (
  '<p>See <a href="https://docs.python.org/3/library/re.html">the re docs</a>'
  ' and <a href="https://regex101.com/">regex101</a>.</p>'
)

# Your code here

### Additional hints

The replacement reuses the groups in the other order: `r"[\2](\1)"`.

`*` is greedy: it repeats as much as it can, and the first `(.*)` runs up to the last `">` of the line, across the first link. Two fixes:

- forbid the double quote in the address and the angle bracket in the text, with negated sets: `[^"]*` and `[^<]*`
- or make the repetition non-greedy, `.*?`, which repeats as little as it can

A skeleton to copy into the exercise cell, replacing each `...`:

```python
print(re.sub(r'<a href="(...)">(...)</a>', r"[\2](\1)", html))
```

### Solution

In [ ]:
# With negated sets
print(re.sub(r'<a href="([^"]*)">([^<]*)</a>', r"[\2](\1)", html))
# With non-greedy repetitions
print(re.sub(r'<a href="(.*?)">(.*?)</a>', r"[\2](\1)", html))

## 4. Quoted strings

In Python, a string is written between `"` or between `'`, and each can contain the other quote. With `re.finditer`, display the content of each string of the line of code `line`, one per line. You should get:

```
Hello
everyone
it's
he said "yes"
```

The pattern `["'](.*?)["']` doesn't work: it stops at the apostrophe of `it's`. Capture the opening quote in a group, and require the same one to close with a backreference.

**Useful documentation**

- [Grouping and backreferences](https://docs.python.org/3/howto/regex.html#grouping)

In [ ]:
line = """print("Hello", 'everyone', "it's", 'he said "yes"')"""

# Your code here

### Additional hints

- in a pattern, `\1` matches exactly the text the first group captured: `(["'])` captures the opening quote, and `\1` requires the same one to close
- the string's content is the second group, `match.group(2)`, a non-greedy repetition `.*?` that stops at the first closing quote (see exercise 3)

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for match in re.finditer(r"""...""", line):
  print(...)
```

### Solution

In [ ]:
for match in re.finditer(r"""(["'])(.*?)\1""", line):
  print(match.group(2))

## 5. A corpus of fraudulent e-mails

The CLAIR corpus gathers 3,976 "Nigerian" scam e-mails (419 fraud), received from 1998 to 2007, one after the other in a single file. The file `debug_file.txt` holds only two of them, to work out the code. Source: Radev (2008), *CLAIR collection of fraud email*, ACL Data and Code Repository, ADCR2008T001, published on https://www.kaggle.com/datasets/rtatman/fraudulent-email-corpus under the CC BY-SA 4.0 license.

Each e-mail starts with a `From r` line, followed by its headers, including:

- `From`: the sender (sometimes blank)
- `Reply-To`: the address to reply to
- `To`: the recipient (sometimes truncated for anonymity)
- `Date`: the date it was sent
- `Subject`: the message's subject
- `Content-Type`: the content's type and its encoding

and finally, after a blank line, the message's body.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh fraud-emails

The files are encoded in `windows-1252`: the next cell reads them into `debug_text` and `work_text`, and displays the start of the corpus.

In [ ]:
import re
from pathlib import Path

debug_text = Path("fraud-emails/debug_file.txt").read_text(encoding="windows-1252")
work_text = Path("fraud-emails/fradulent_emails.txt").read_text(encoding="windows-1252")
print(work_text[:1500])

### 5.1 Splitting the corpus into e-mails

Write `split_emails(text)`, which returns the list of the e-mails of `text`: split it with `re.split` at each line that starts with `From r`. Test it on `debug_text`, which must give 2 e-mails, then store those of `work_text` in `emails`: 3,976.

**Useful documentation**

- [`re.split`](https://docs.python.org/3/library/re.html#re.split)

In [ ]:
def split_emails(text: str) -> list[str]: ...


# Your code here

#### Additional hints

- `^` matches every start of line only with the `re.MULTILINE` flag, to pass by name, `flags=` (passing it positionally is deprecated since Python 3.13)
- the file starts with a `From r` line: the first piece `re.split` returns, what comes before it, is an empty string, to drop with `[1:]`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def split_emails(text: str) -> list[str]:
  return re.split(r"...", text, flags=...)[1:]


print(len(split_emails(debug_text)))
emails = split_emails(work_text)
print(len(emails))
```

#### Solution

In [ ]:
def split_emails(text: str) -> list[str]:
  return re.split(r"^From r", text, flags=re.MULTILINE)[1:]


print(len(split_emails(debug_text)))
emails = split_emails(work_text)
print(len(emails))

### 5.2 The first ten senders

With `re.findall`, display the first ten `From:` header lines of the corpus. They start with `From:`, with a colon, unlike the `From r` lines that separate the e-mails. The first one is:

```
From: "MR. JAMES NGOLA." <james_ngola2002@maktoob.com>
```

In [ ]:
# Your code here

#### Additional hints

- `^From:.*$` matches a whole header line: `.` doesn't match the line break, so `.*` stops at the end of the line
- `flags=re.MULTILINE` again, so that `^` and `$` match every start and every end of line
- `re.findall` returns a list, of which `[:10]` keeps the first ten items

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for line in re.findall(r"...", work_text, flags=...)[:10]:
  print(line)
```

#### Solution

In [ ]:
for line in re.findall(r"^From:.*$", work_text, flags=re.MULTILINE)[:10]:
  print(line)

### 5.3 Senders and addresses

A `From:` line often has the form `From: "Mr. Ben Suleman" <bensul2004nng@spinfinder.com>`: a name, sometimes in quotes, then an address in angle brackets.

1. Write `process_name(name)`, which returns `name` without its `'` and `"`, and without its spaces at the start and at the end: `process_name(' "Mr. Ben Suleman" ')` returns `'Mr. Ben Suleman'`.
2. Build the dictionary `senders`, which maps each sender's name, passed through `process_name`, to the set of their addresses. Keep the lines of the form `name <address>` whose address is plausible (non-blank characters, an `@`, then a domain with a period) and whose name isn't empty; skip the others, which have only an address or only a name.
3. Display the number of `From:` lines kept and skipped, the number of senders, and the number of senders with several addresses. With the pattern of the hints, you should find 3,190 lines kept, 598 skipped, and 2,407 senders, 162 of whom have several addresses.

In [ ]:
def process_name(name: str) -> str: ...


# Your code here

#### Additional hints

- in `process_name`, `re.sub` with the set `['"]` removes the quotes, and the strings' `strip` method the spaces at the start and at the end
- one pattern for the whole line, with a group for the name and one for the address: `^From:(.*)<(\w\S*@\S+\.\w+)>$`, again with `re.MULTILINE`
- `re.finditer` gives each line matched, of which `match.group(1)` and `match.group(2)` are the name and the address
- the number of `From:` lines: the length of the list `re.findall(r"^From:", work_text, flags=re.MULTILINE)` returns

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def process_name(name: str) -> str:
  return re.sub(..., "", name).strip()


senders = {}
kept = 0
for match in re.finditer(r"...", work_text, flags=re.MULTILINE):
  name = process_name(match.group(1))
  if name == "":
    continue
  kept += 1
  if name not in senders:
    senders[name] = set()
  senders[name].add(...)

total = len(re.findall(r"^From:", work_text, flags=re.MULTILINE))
several = [name for name, addresses in senders.items() if len(addresses) > 1]
print(f"{kept} lines kept, {total - kept} skipped")
print(f"{len(senders)} senders, {len(several)} of whom have several addresses")
```

#### Solution

In [ ]:
def process_name(name: str) -> str:
  return re.sub(r"['\"]", "", name).strip()


print(repr(process_name(' "Mr. Ben Suleman" ')))

senders = {}
kept = 0
for match in re.finditer(
  r"^From:(.*)<(\w\S*@\S+\.\w+)>$", work_text, flags=re.MULTILINE
):
  name = process_name(match.group(1))
  if name == "":
    continue
  kept += 1
  if name not in senders:
    senders[name] = set()
  senders[name].add(match.group(2))

total = len(re.findall(r"^From:", work_text, flags=re.MULTILINE))
several = [name for name, addresses in senders.items() if len(addresses) > 1]
print(f"{kept} lines kept, {total - kept} skipped")
print(f"{len(senders)} senders, {len(several)} of whom have several addresses")

### 5.4 E-mails per year

For each e-mail of `emails`, find the year of its first `Date:` header with `re.search` and a named group `year`, and count the e-mails of each year in the dictionary `per_year`. Display the years in order, each with its number of e-mails, then the number of e-mails without a year. You should find 1,170 e-mails in 2006, the busiest year, and 566 without a year.

The corpus spans 1998 to 2007: where do the other years come from?

**Useful documentation**

- [Named groups](https://docs.python.org/3/howto/regex.html#non-capturing-and-named-groups)

In [ ]:
# Your code here

#### Additional hints

- a named group is written `(?P<year>...)`, and `match.group("year")` gives what it captured
- in `^Date:.*?(?P<year>\d{4})`, the non-greedy repetition stops at the first four-digit number of the line: `2002` in `Date: Thu, 31 Oct 2002 05:10:00`
- `re.search` returns the first match, or `None` when the e-mail has no `Date:` header with a year
- `per_year.get(year, 0)` is `0` for a year not seen yet, and `sorted(per_year.items())` gives the `(year, count)` pairs in the order of the years

A skeleton to copy into the exercise cell, replacing each `...`:

```python
per_year = {}
missing = 0
for email in emails:
  match = re.search(r"...", email, flags=re.MULTILINE)
  if match is None:
    missing += 1
  else:
    year = ...
    per_year[year] = per_year.get(year, 0) + 1

for year, count in sorted(per_year.items()):
  print(year, count)
print("Without a year:", missing)
```

#### Solution

In [ ]:
per_year = {}
missing = 0
for email in emails:
  match = re.search(r"^Date:.*?(?P<year>\d{4})", email, flags=re.MULTILINE)
  if match is None:
    missing += 1
  else:
    year = match.group("year")
    per_year[year] = per_year.get(year, 0) + 1

for year, count in sorted(per_year.items()):
  print(year, count)
print("Without a year:", missing)

# The 15 e-mails dated 1980, 1990, 2015 or 2082 come from computers with a wrong clock
# (or from forged headers): a per-year analysis leaves them out.